# 07 - CI/CD with SageMaker Pipelines

This notebook satisfies the Week 6 project step: **implement a CI/CD pipeline to automate training, evaluation, and deployment.**

The pipeline DAG (modeled on Lab 6.1) is:

1. **Preprocess** (`src/preprocessing.py`) - validates the split CSVs and prepares train/validation/test channels plus the batch-transform input. This is the *model code / system integration checkpoint*: bad data or schema stops the pipeline here.
2. **Train** (`src/train_sklearn.py`) - trains the TF-IDF + Logistic Regression model **with the training data** (validation channel for reference metrics).
3. **Evaluate** (`src/evaluation.py`) - scores the model **with the testing data** and writes `evaluation.json`. This is the *model performance checkpoint*.
4. **ConditionStep** - quality gate: macro F1 >= threshold (default 0.80).
   - **Pass:** register the model in the SageMaker Model Registry, create the model, and run Batch Transform on the reserved production reviews (deployment).
   - **Fail:** `FailStep` marks the execution as failed and blocks deployment.

We run the pipeline twice: once with the baseline hyperparameters and once with improved hyperparameters, satisfying "improve your initial model and run it through your CI/CD pipeline".


## 0. Install SageMaker SDK for Pipelines

SageMaker Pipelines requires the SageMaker Python SDK v2. Run the next cell first in SageMaker Studio. If it installs or upgrades the package, restart the kernel once, then run the notebook from the top.


In [3]:
# %pip install --upgrade "sagemaker>=2,<3"
import sys
!{sys.executable} -m pip install "sagemaker<3.0.0" --upgrade

  Using cached sagemaker-2.257.3-py3-none-any.whl.metadata (17 kB)


  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)


  Using cached pathos-0.3.5-py3-none-any.whl.metadata (11 kB)
  Using cached sagemaker_core-1.0.78-py3-none-any.whl.metadata (4.9 kB)


  Using cached ppft-1.7.8-py3-none-any.whl.metadata (12 kB)
  Using cached pox-0.3.7-py3-none-any.whl.metadata (8.0 kB)


Using cached sagemaker-2.257.3-py3-none-any.whl (1.7 MB)
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached packaging-24.2-py3-none-any.whl (65 kB)
Using cached sagemaker_core-1.0.78-py3-none-any.whl (444 kB)
Using cached pathos-0.3.5-py3-none-any.whl (82 kB)
Using cached pox-0.3.7-py3-none-any.whl (29 kB)
Using cached ppft-1.7.8-py3-none-any.whl (56 kB)


...

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 6/7 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 6/7 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 6/7 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━ 6/7 [sagemaker]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [sagemaker]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
autogluon-timeseries 1.5.0 requires peft<0.18,>=0.13.0, which is not installed.
skops 0.14.0 requires prettytable>=3.9, which is not installed.
autogluon-multimodal 1.5.0 requires fsspec[http]<=2025.3, but you have fsspec 2026.3.0 which is incompatible.
sagemaker-mlops 1.7.1 requires sagemaker-core>=2.7.1, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-serve 1.9.0 requires sagemaker-core>=2.9.0, but you have sagemaker-core 1.0.78 which is incompatible.
sagemaker-studio-analytics-ex

In [1]:
import sagemaker
print(sagemaker.__version__)
print(sagemaker.__file__)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


2.257.3
/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py


In [2]:
import json

import boto3
import sagemaker
from sagemaker.inputs import TrainingInput, TransformInput
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.model_metrics import MetricsSource, ModelMetrics
from sagemaker.transformer import Transformer
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.fail_step import FailStep
from sagemaker.workflow.functions import Join, JsonGet
from sagemaker.workflow.model_step import ModelStep
from sagemaker.workflow.parameters import ParameterFloat, ParameterInteger, ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import PipelineSession
from sagemaker.workflow.properties import PropertyFile
from sagemaker.workflow.steps import ProcessingStep, TrainingStep, TransformStep

print("SageMaker SDK:", sagemaker.__version__)


SageMaker SDK: 2.257.3


## 1. Configure AWS, role, and S3 paths


In [3]:
%store -r bucket
%store -r role
%store -r region

boto_session = boto3.Session()
sts = boto3.client("sts")
identity = sts.get_caller_identity()
account_id = identity["Account"]
region = globals().get("region") or boto_session.region_name

if globals().get("role"):
    role = globals()["role"]
elif hasattr(sagemaker, "get_execution_role"):
    role = sagemaker.get_execution_role()
else:
    caller_arn = identity["Arn"]
    if ":assumed-role/" in caller_arn:
        role_name = caller_arn.split(":assumed-role/", 1)[1].split("/", 1)[0]
        role = f"arn:aws:iam::{account_id}:role/{role_name}"
    else:
        role = caller_arn

bucket = globals().get("bucket") or f"yelp-sentiment-mlops-{account_id}"

pipeline_session = PipelineSession(default_bucket=bucket)
sagemaker_client = boto3.client("sagemaker")

pipeline_name = "yelp-sentiment-cicd-pipeline"
model_package_group_name = "yelp-sentiment-models"
splits_s3 = f"s3://{bucket}/processed/splits/"
model_output_s3 = f"s3://{bucket}/models/cicd-pipeline"
transform_output_s3 = f"s3://{bucket}/batch/cicd-pipeline/output/"

print("Region:", region)
print("Bucket:", bucket)
print("Role:", role)
print("Pipeline:", pipeline_name)


no stored variable or alias role


Region: us-east-1
Bucket: yelp-sentiment-mlops-965705611982
Role: arn:aws:iam::965705611982:role/LabRole
Pipeline: yelp-sentiment-cicd-pipeline


## 2. Pipeline parameters

Parameters let us rerun the same pipeline with different hyperparameters or thresholds without changing code - this is how we push the improved model through the same CI/CD checkpoints.


In [4]:
processing_instance_count = ParameterInteger(name="ProcessingInstanceCount", default_value=1)
instance_type_param = ParameterString(name="InstanceType", default_value="ml.m5.large")
model_approval_status = ParameterString(name="ModelApprovalStatus", default_value="PendingManualApproval")
input_data = ParameterString(name="InputData", default_value=splits_s3)
f1_threshold = ParameterFloat(name="F1Threshold", default_value=0.80)

max_features_param = ParameterInteger(name="MaxFeatures", default_value=50000)
ngram_max_param = ParameterInteger(name="NgramMax", default_value=2)
reg_c_param = ParameterFloat(name="RegC", default_value=1.0)


## 3. Preprocessing step (data validation checkpoint)


In [5]:
sklearn_processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type="ml.m5.large",
    instance_count=processing_instance_count,
    base_job_name="yelp-sentiment-preprocess",
    role=role,
    sagemaker_session=pipeline_session,
)

processor_args = sklearn_processor.run(
    inputs=[ProcessingInput(source=input_data, destination="/opt/ml/processing/input")],
    outputs=[
        ProcessingOutput(output_name="train", source="/opt/ml/processing/train"),
        ProcessingOutput(output_name="validation", source="/opt/ml/processing/validation"),
        ProcessingOutput(output_name="test", source="/opt/ml/processing/test"),
        ProcessingOutput(output_name="batch", source="/opt/ml/processing/batch"),
    ],
    code="../src/preprocessing.py",
)

step_process = ProcessingStep(name="YelpPreprocess", step_args=processor_args)


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


/opt/conda/lib/python3.12/site-packages/sagemaker/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


## 4. Training step (trains with the training data)


In [7]:
estimator = SKLearn(
    entry_point="train_sklearn.py",
    source_dir="../src",
    role=role,
    instance_type=instance_type_param,
    instance_count=1,
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=pipeline_session,
    output_path=model_output_s3,
    hyperparameters={
        "max-features": max_features_param,
        "ngram-max": ngram_max_param,
        "c": reg_c_param,
        "max-iter": 1000,
    },
)

train_args = estimator.fit(
    inputs={
        "train": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri,
            content_type="text/csv",
        ),
        "validation": TrainingInput(
            s3_data=step_process.properties.ProcessingOutputConfig.Outputs["validation"].S3Output.S3Uri,
            content_type="text/csv",
        ),
    }
)

step_train = TrainingStep(name="YelpTrain", step_args=train_args)


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


## 5. Evaluation step (evaluates with the testing data)

Writes `evaluation.json`; the ConditionStep reads `classification_metrics.f1_macro.value` from it.


In [8]:
eval_processor = SKLearnProcessor(
    framework_version="1.2-1",
    instance_type="ml.m5.large",
    instance_count=1,
    base_job_name="yelp-sentiment-eval",
    role=role,
    sagemaker_session=pipeline_session,
)

eval_args = eval_processor.run(
    inputs=[
        ProcessingInput(
            source=step_train.properties.ModelArtifacts.S3ModelArtifacts,
            destination="/opt/ml/processing/model",
        ),
        ProcessingInput(
            source=step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri,
            destination="/opt/ml/processing/test",
        ),
    ],
    outputs=[ProcessingOutput(output_name="evaluation", source="/opt/ml/processing/evaluation")],
    code="../src/evaluation.py",
)

evaluation_report = PropertyFile(name="EvaluationReport", output_name="evaluation", path="evaluation.json")
step_eval = ProcessingStep(name="YelpEvaluate", step_args=eval_args, property_files=[evaluation_report])


INFO:sagemaker.image_uris:Defaulting to only available Python version: py3


## 6. Register, create model, and Batch Transform steps (deployment on pass)


In [9]:
model = SKLearnModel(
    model_data=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    role=role,
    entry_point="inference.py",
    source_dir="../src",
    framework_version="1.2-1",
    py_version="py3",
    sagemaker_session=pipeline_session,
)

model_metrics = ModelMetrics(
    model_statistics=MetricsSource(
        s3_uri=Join(
            on="/",
            values=[
                step_eval.properties.ProcessingOutputConfig.Outputs["evaluation"].S3Output.S3Uri,
                "evaluation.json",
            ],
        ),
        content_type="application/json",
    )
)

register_args = model.register(
    content_types=["text/plain", "text/csv"],
    response_types=["application/json"],
    inference_instances=["ml.m5.large"],
    transform_instances=["ml.m5.large"],
    model_package_group_name=model_package_group_name,
    approval_status=model_approval_status,
    model_metrics=model_metrics,
)
step_register = ModelStep(name="YelpRegisterModel", step_args=register_args)

step_create_model = ModelStep(
    name="YelpCreateModel",
    step_args=model.create(instance_type="ml.m5.large"),
)

transformer = Transformer(
    model_name=step_create_model.properties.ModelName,
    instance_type="ml.m5.large",
    instance_count=1,
    output_path=transform_output_s3,
    accept="application/json",
    assemble_with="Line",
    sagemaker_session=pipeline_session,
)

step_transform = TransformStep(
    name="YelpBatchTransform",
    transformer=transformer,
    inputs=TransformInput(
        data=step_process.properties.ProcessingOutputConfig.Outputs["batch"].S3Output.S3Uri,
        content_type="text/plain",
        split_type="Line",
    ),
)


## 7. Quality gate: ConditionStep + FailStep


In [12]:
step_fail = FailStep(
    name="YelpF1Fail",
    error_message=Join(on=" ", values=["Execution failed: macro F1 below threshold", f1_threshold]),
)

cond_gte = ConditionGreaterThanOrEqualTo(
    left=JsonGet(
        step_name=step_eval.name,
        property_file=evaluation_report,
        json_path="classification_metrics.f1_macro.value",
    ),
    right=f1_threshold,
)

step_cond = ConditionStep(
    name="YelpF1Gate",
    conditions=[cond_gte],
    if_steps=[step_register, step_create_model, step_transform],
    else_steps=[step_fail],
)


## 8. Assemble and upsert the pipeline


In [14]:
import os

print("Current directory:", os.getcwd())
print("Notebook sees src directory:", os.path.exists("../src"))
print("Notebook sees preprocessing.py:", os.path.exists("../src/preprocessing.py"))

Current directory: /home/sagemaker-user/yelp-sentiment-mlops-pipeline/notebooks
Notebook sees src directory: True
Notebook sees preprocessing.py: False


In [15]:
import os

print(os.listdir("../src"))

['inference.py', 'train_sklearn.py']


In [16]:
import os

for root, dirs, files in os.walk(".."):
    for file in files:
        if "preprocess" in file.lower() or "evaluat" in file.lower():
            print(os.path.join(root, file))

../notebooks/05_model_training_evaluation_deployment.ipynb
../notebooks/.ipynb_checkpoints/05_model_training_evaluation_deployment-checkpoint.ipynb
../reports/model_evaluation_summary.md


In [17]:
pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        processing_instance_count,
        instance_type_param,
        model_approval_status,
        input_data,
        f1_threshold,
        max_features_param,
        ngram_max_param,
        reg_c_param,
    ],
    steps=[step_process, step_train, step_eval, step_cond],
    sagemaker_session=pipeline_session,
)

pipeline.upsert(role_arn=role)
definition = json.loads(pipeline.definition())
print("Pipeline steps:", [step["Name"] for step in definition["Steps"]])


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Pipeline steps: ['YelpPreprocess', 'YelpTrain', 'YelpEvaluate', 'YelpF1Gate']


## 9. Execution 1 - baseline model through the CI/CD pipeline

This takes roughly 15-25 minutes. The pipeline view in SageMaker Studio shows the DAG and per-step status.


In [18]:
execution_baseline = pipeline.start()
print("Started:", execution_baseline.arn)
execution_baseline.wait(delay=60, max_attempts=120)
baseline_steps = execution_baseline.list_steps()
for step in baseline_steps:
    print(f"{step['StepName']:25s} {step['StepStatus']}")


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Started: arn:aws:sagemaker:us-east-1:965705611982:pipeline/yelp-sentiment-cicd-pipeline/execution/mew2kx8cndwk


YelpBatchTransform        Succeeded
YelpCreateModel-CreateModel Succeeded
YelpRegisterModel-RegisterModel Succeeded
YelpF1Gate                Succeeded
YelpEvaluate              Succeeded
YelpTrain                 Succeeded
YelpPreprocess            Succeeded


## 10. Execution 2 - improved model through the same pipeline

Improved hyperparameters: larger vocabulary (100k features), trigrams, and stronger C. Same checkpoints, no code changes - only parameters.


In [19]:
execution_improved = pipeline.start(
    parameters=dict(
        MaxFeatures=100000,
        NgramMax=3,
        RegC=2.0,
    )
)
print("Started:", execution_improved.arn)
execution_improved.wait(delay=60, max_attempts=120)
improved_steps = execution_improved.list_steps()
for step in improved_steps:
    print(f"{step['StepName']:25s} {step['StepStatus']}")


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Started: arn:aws:sagemaker:us-east-1:965705611982:pipeline/yelp-sentiment-cicd-pipeline/execution/agmosts2poa9


YelpBatchTransform        Succeeded
YelpCreateModel-CreateModel Succeeded
YelpRegisterModel-RegisterModel Succeeded
YelpF1Gate                Succeeded
YelpEvaluate              Succeeded
YelpTrain                 Succeeded
YelpPreprocess            Succeeded


## 11. Demonstrate the failure path

Setting an impossible threshold routes the execution to the FailStep, proving the gate blocks bad models. Uncomment to run.


In [15]:
execution_fail_demo = pipeline.start(parameters=dict(F1Threshold=0.99))
try:
    execution_fail_demo.wait(delay=60, max_attempts=120)
except Exception as exc:
    print("Expected failure:", exc)
for step in execution_fail_demo.list_steps():
    print(f"{step['StepName']:25s} {step['StepStatus']}")


INFO:sagemaker.telemetry.telemetry_logging:SageMaker Python SDK will collect telemetry to help us better understand our user's needs, diagnose issues, and deliver additional features.
To opt out of telemetry, please disable via TelemetryOptOut parameter in SDK defaults config. For more information, refer to https://sagemaker.readthedocs.io/en/stable/overview.html#configuring-and-using-defaults-with-the-sagemaker-python-sdk.


Expected failure: Waiter PipelineExecutionComplete failed: Waiter encountered a terminal failure state: For expression "PipelineExecutionStatus" we matched expected path: "Failed"
YelpF1Fail                Failed
YelpF1Gate                Succeeded
YelpEvaluate              Succeeded
YelpTrain                 Succeeded
YelpPreprocess            Succeeded


## 12. Collect results and write the Week 6 report


In [20]:
def get_evaluation_metrics(steps):
    """Fetch evaluation.json metrics from an execution's YelpEvaluate processing job."""
    for step in steps:
        if step["StepName"] != "YelpEvaluate":
            continue
        job_arn = step["Metadata"]["ProcessingJob"]["Arn"]
        job_name = job_arn.split("/")[-1]
        description = sagemaker_client.describe_processing_job(ProcessingJobName=job_name)
        for output in description["ProcessingOutputConfig"]["Outputs"]:
            if output["OutputName"] == "evaluation":
                s3_uri = output["S3Output"]["S3Uri"] + "/evaluation.json"
                bucket_name, key = s3_uri.replace("s3://", "", 1).split("/", 1)
                body = boto3.client("s3").get_object(Bucket=bucket_name, Key=key)["Body"].read()
                return json.loads(body)["classification_metrics"]
    return None

baseline_metrics = get_evaluation_metrics(baseline_steps)
improved_metrics = get_evaluation_metrics(improved_steps)

packages = sagemaker_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name,
    SortBy="CreationTime",
    SortOrder="Descending",
).get("ModelPackageSummaryList", [])

print("Baseline macro F1:", baseline_metrics["f1_macro"]["value"])
print("Improved macro F1:", improved_metrics["f1_macro"]["value"])
print("Registered model packages:", len(packages))


Baseline macro F1: 0.9467946794679467
Improved macro F1: 0.9481950246501674
Registered model packages: 2


In [21]:
from pathlib import Path

report_dir = Path("../reports")
report_dir.mkdir(parents=True, exist_ok=True)

def step_lines(steps):
    return [f"| {s['StepName']} | {s['StepStatus']} |" for s in reversed(steps)]

lines = [
    "# Week 6 CI/CD Pipeline Summary",
    "",
    f"Pipeline: `{pipeline_name}`",
    f"Model Package Group: `{model_package_group_name}`",
    f"Quality gate: macro F1 >= 0.80 on the test split (ConditionStep + FailStep)",
    "",
    "## Checkpoints",
    "",
    "- Data/system integration: `YelpPreprocess` validates schema and splits before training.",
    "- Model code: training and inference scripts run inside pipeline containers from version-controlled `src/`.",
    "- Model performance: `YelpEvaluate` scores the test split; `YelpF1Gate` blocks deployment below threshold.",
    "- Deployment: on pass, the model is registered and Batch Transform scores the production reviews.",
    "",
    "## Execution 1 - Baseline (max_features=50000, ngram<=2, C=1.0)",
    "",
    f"- ARN: `{execution_baseline.arn}`",
    f"- Test macro F1: {baseline_metrics['f1_macro']['value']:.4f}",
    f"- Test accuracy: {baseline_metrics['accuracy']['value']:.4f}",
    "",
    "| Step | Status |",
    "|---|---|",
    *step_lines(baseline_steps),
    "",
    "## Execution 2 - Improved (max_features=100000, ngram<=3, C=2.0)",
    "",
    f"- ARN: `{execution_improved.arn}`",
    f"- Test macro F1: {improved_metrics['f1_macro']['value']:.4f}",
    f"- Test accuracy: {improved_metrics['accuracy']['value']:.4f}",
    "",
    "| Step | Status |",
    "|---|---|",
    *step_lines(improved_steps),
    "",
    "## Model Registry",
    "",
]
for package in packages[:5]:
    lines.append(f"- `{package['ModelPackageArn']}` ({package['ModelApprovalStatus']})")

lines += [
    "",
    "## Artifacts",
    "",
    f"- Batch Transform output: `{transform_output_s3}`",
    f"- Model artifacts: `{model_output_s3}`",
]

report_path = report_dir / "cicd_pipeline_summary.md"
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("Wrote", report_path)
print("\n".join(lines))


Wrote ../reports/cicd_pipeline_summary.md
# Week 6 CI/CD Pipeline Summary

Pipeline: `yelp-sentiment-cicd-pipeline`
Model Package Group: `yelp-sentiment-models`
Quality gate: macro F1 >= 0.80 on the test split (ConditionStep + FailStep)

## Checkpoints

- Data/system integration: `YelpPreprocess` validates schema and splits before training.
- Model code: training and inference scripts run inside pipeline containers from version-controlled `src/`.
- Model performance: `YelpEvaluate` scores the test split; `YelpF1Gate` blocks deployment below threshold.
- Deployment: on pass, the model is registered and Batch Transform scores the production reviews.

## Execution 1 - Baseline (max_features=50000, ngram<=2, C=1.0)

- ARN: `arn:aws:sagemaker:us-east-1:965705611982:pipeline/yelp-sentiment-cicd-pipeline/execution/mew2kx8cndwk`
- Test macro F1: 0.9468
- Test accuracy: 0.9468

| Step | Status |
|---|---|
| YelpPreprocess | Succeeded |
| YelpTrain | Succeeded |
| YelpEvaluate | Succeeded |
| Ye